In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from simulators import NestedModelFamily, ContextManager
from simulators.benchmarks import DDM
from adapters import Adapter

# Metas

In [4]:
intrinsic_params = ["v", "a", "tau", "s_v", "s_tau", "decay"]

# Priors

In [5]:
priors = {
    "v":     {"intercept": lambda: np.random.gamma(3.0, 0.8),
              "slope":     lambda: np.random.normal(0.0, 3.0)},
    "a":     {"intercept": lambda: np.random.gamma(10.0, 0.3),
              "slope":     lambda: np.random.normal(0.0, 1.0)},
    "tau":   {"intercept": lambda: np.random.gamma(3.0, 0.2),
              "slope":     lambda: 0.0},
    "s_v":   {"intercept": lambda: np.random.gamma(1.0, 0.2),
              "slope":     lambda: 0.0},
    "s_tau": {"intercept": lambda: np.random.uniform(0.0, 0.4),
              "slope":     lambda: 0.0},
    "decay": {"intercept": lambda: np.random.gamma(1.0, 0.4),
              "slope":     lambda: 0.0},
}

# Context Manager

In [6]:
context_manager = ContextManager()

# Model Family

In [7]:
model_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    context_manager=context_manager,
    prior_fun=priors,
    intrinsic_params=intrinsic_params,
)

In [17]:
samples = model_family.batch_sample(
    batch_size=3,
    mask_randomizer_kwargs=dict(
        free_intrinsics={"v", "a", "tau", "s_v", "decay"},
        fixed_intrinsics={"s_tau"}
    ),
    min_num_obs=20,
    max_num_obs=500
)

In [18]:
samples["param_masks"].shape

(3, 108)

In [19]:
samples["param_matrices"].shape

(3, 108)

In [20]:
samples["regressor_masks"].shape

(3, 18)

In [21]:
samples

{'model_names': ['DDM', 'DDM', 'DDM'],
 'design_configs': [{'u_0': ['decay']},
  {'u_0': ['decay'],
   'u_1': ['a', 's_v', 'decay'],
   'u_2': ['decay'],
   'u_3': [],
   'u_4': ['v', 'a', 'decay']},
  {'u_0': ['a', 'decay'],
   'u_1': ['decay'],
   'u_2': ['decay'],
   'u_3': ['v', 'a', 's_v', 'decay'],
   'u_4': ['v', 'a', 'tau'],
   'u_5': ['decay']}],
 'design_matrices': array([[[0.02903825, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.17937073, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.89791819, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ]],
 
        [[0.602168

# Adapter

In [14]:
adapter = Adapter()

In [15]:
design_matrices = adapter.convert_dtype(samples["design_matrices"], dtype=np.float32)
param_masks = adapter.convert_dtype(samples["param_masks"], dtype=np.float32)
rts = adapter.convert_dtype(samples["sim_data"]["rts"], dtype=np.float32)
choices = adapter.convert_dtype(samples["sim_data"]["choices"], dtype=np.float32)

In [16]:
batch_size, num_obs, num_cols = design_matrices.shape
print(batch_size, num_obs, num_cols)

3 376 18


In [17]:
y_rts_col = adapter.atleast_2d(rts, orientation="col")                 # (N, 1)
y_ch_col  = adapter.atleast_2d(rts,  orientation="col")

In [18]:
sim_data = adapter.concatenate([y_rts_col, y_ch_col], axis=1, dtype=np.float32, pad=False)